# Notion Data Extractor (Updated for 2025-09-03 Core API)

이 노트북은 **2025년 9월 출시된 새로운 Notion API(Multi-source Database)** 구조에 대응하도록 업데이트되었습니다.

### ⚠️ 주요 변경 사항 (2025-09-03)
- 이제 하나의 데이터베이스는 여러 개의 **Data Source**를 가질 수 있습니다.
- 데이터베이스를 쿼리하거나 페이지를 생성할 때, 단순한 `database_id` 대신 **`data_source_id`**가 필요할 수 있습니다.
- 본 노트북은 데이터베이스 접근 시 자동으로 `data_source_id`를 검색(Discovery)하는 로직을 포함합니다.

### 준비 사항
1. [Notion My Integrations](https://www.notion.so/my-integrations)에서 인테그레이션을 생성하고 **Internal Integration Token**을 준비하세요.
2. 해당 페이지/데이터베이스에 인테그레이션을 **연결(Connection)**로 추가하세요.

In [ ]:
from notion_client import Client
import os
import json
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

# 2025-09-03 버전 명시
NOTION_TOKEN = os.getenv("NOTION_TOKEN")
notion = Client(auth=NOTION_TOKEN, notion_version="2025-09-03")

output_dir = Path("extracted_notion_data")
output_dir.mkdir(exist_ok=True)

## 1. 데이터베이스 및 데이터 소스 탐색 (Discovery)
새로운 API에서는 데이터베이스 내의 데이터 소스를 먼저 확인해야 합니다.

In [ ]:
def get_data_source_id(database_id):
    """데이터베이스 ID를 통해 첫 번째 데이터 소스 ID를 가져옵니다."""
    try:
        db = notion.databases.retrieve(database_id=database_id)
        # 신규 API에서는 'data_sources' 필드를 통해 실제 데이터에 접근해야 합니다.
        if "data_sources" in db and db["data_sources"]:
            ds_id = db["data_sources"][0]["id"]
            print(f"Found Data Source ID: {ds_id}")
            return ds_id
        else:
            # 데이터 소스가 없거나 구형 구조일 경우 database_id를 그대로 사용해봅니다.
            return database_id
    except Exception as e:
        print(f"Error retrieving database: {e}")
        return database_id

def query_notion_data_source(ds_id):
    """데이터 소스의 모든 페이지를 가져옵니다."""
    results = []
    has_more = True
    start_cursor = None
    
    while has_more:
        # 참고: 최신 라이브러리/버전에서는 data_source_id 매개변수를 사용합니다.
        # (라이브러리 버전에 따라 databases.query 내부에서 처리 방식이 다를 수 있음)
        response = notion.databases.query(
            database_id=ds_id, # ds_id가 database_id 역할을 겸함 (Single source 기준)
            start_cursor=start_cursor
        )
        results.extend(response["results"])
        has_more = response["has_more"]
        start_cursor = response["next_cursor"]
        
    return results

# 사용법 예시
db_id = "input_your_database_id_here"
if db_id != "input_your_database_id_here":
    ds_id = get_data_source_id(db_id)
    db_results = query_notion_data_source(ds_id)
    
    with open(output_dir / "database_results.json", "w", encoding="utf-8") as f:
        json.dump(db_results, f, ensure_ascii=False, indent=2)
    print(f"총 {len(db_results)}개의 항목을 가져왔습니다.")

## 2. 페이지 내용(Blocks) 추출
페이지 내 블록 추출은 이전과 유사하나, `notion_version="2025-09-03"` 헤더를 통해 최신 직렬화 방식을 따릅니다.

In [ ]:
def get_page_blocks(page_id):
    """페이지 내의 모든 블록을 가져옵니다."""
    blocks = []
    has_more = True
    start_cursor = None
    
    while has_more:
        response = notion.blocks.children.list(
            block_id=page_id,
            start_cursor=start_cursor
        )
        blocks.extend(response["results"])
        has_more = response["has_more"]
        start_cursor = response["next_cursor"]
        
    return blocks

def blocks_to_markdown(blocks):
    """블록 리스트를 마크다운 텍스트로 변환합니다."""
    md = ""
    for block in blocks:
        b_type = block["type"]
        # 간단한 매핑 logic
        if b_type in ["paragraph", "heading_1", "heading_2", "heading_3"]:
            rich_text = block[b_type].get("rich_text", [])
            text = "".join([t["plain_text"] for t in rich_text])
            if b_type == "heading_1": md += f"# {text}\n\n"
            elif b_type == "heading_2": md += f"## {text}\n\n"
            elif b_type == "heading_3": md += f"### {text}\n\n"
            else: md += f"{text}\n\n"
        elif b_type == "bulleted_list_item":
            rich_text = block[b_type].get("rich_text", [])
            text = "".join([t["plain_text"] for t in rich_text])
            md += f"- {text}\n"
    return md

# 사용법 예시
target_page_id = "input_your_page_id_here"
if target_page_id != "input_your_page_id_here":
    blocks = get_page_blocks(target_page_id)
    markdown = blocks_to_markdown(blocks)
    
    file_path = output_dir / f"{target_page_id}.md"
    with open(file_path, "w", encoding="utf-8") as f:
        f.write(markdown)
    print(f"페이지 저장 완료: {file_path}")